In [0]:
%sql
WITH tb_pedidos AS (
    SELECT
        *
    FROM workspace.olist.orders
    WHERE order_purchase_timestamp < '2018-07-01'
),
tb_pedidos_seller AS (
    SELECT p.order_id,
        DATE(p.order_purchase_timestamp) as order_purchase_timestamp,
        s.seller_id,
        sum(i.price) as receita
    FROM workspace.olist.sellers AS s 
    LEFT
    JOIN tb_pedidos AS p
        ON i.seller_id = s.seller_id
    LEFT
    JOIN workspace.olist.order_items AS i
        ON p.order_id = i.order_id
    WHERE s.seller_id IS NOT NULL
      AND p.order_status NOT IN ('canceled', 'invoiced')
    GROUP 
      BY 1,2,3
),
tb_intervalos AS (
    SELECT
        DATE '2018-07-01' AS DTLIMITE,
        DATE '2018-07-01' - INTERVAL '28' DAY AS D28,
        DATE '2018-07-01' - INTERVAL '56' DAY AS D56,
        DATE '2018-07-01' - INTERVAL '365' DAY AS D365
),
tb_seller_rfv AS (
    SELECT
        ps.seller_id,
        -- Receita por período
        SUM(
            CASE
                WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D28 AND inter.DTLIMITE
                THEN ps.receita
                ELSE 0
            END
        ) AS vlReceitaP28,

        SUM(
            CASE
                WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D56 AND inter.DTLIMITE
                THEN ps.receita
                ELSE 0
            END
        ) AS vlReceitaP56,

        SUM(
            CASE
                WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D365 AND inter.DTLIMITE
                THEN ps.receita
                ELSE 0
            END
        ) AS vlReceitaP365,

        -- Ticket médio = receita / número de pedidos
        SUM(
            CASE
                WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D28 AND inter.DTLIMITE
                THEN ps.receita
                ELSE 0
            END
        ) /
        NULLIF(
            COUNT(
                CASE
                    WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D28 AND inter.DTLIMITE
                    THEN ps.order_id
                    ELSE NULL
                END
            ),
            0
        ) AS vlTicketMedioP28,

        SUM(
            CASE
                WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D56 AND inter.DTLIMITE
                THEN ps.receita
                ELSE 0
            END
        ) /
        NULLIF(
            COUNT(
                CASE
                    WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D56 AND inter.DTLIMITE
                    THEN ps.order_id
                    ELSE NULL
                END
            ),
            0
        ) AS vlTicketMedioP56,

        SUM(
            CASE
                WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D365 AND inter.DTLIMITE
                THEN ps.receita
                ELSE 0
            END
        ) /
        NULLIF(
            COUNT(
                CASE
                    WHEN DATE(ps.order_purchase_timestamp) BETWEEN inter.D365 AND inter.DTLIMITE
                    THEN ps.order_id
                    ELSE NULL
                END
            ),
            0
        ) AS vlTicketMedioP365

    FROM tb_pedidos_seller AS ps
    CROSS JOIN tb_intervalos AS inter
    GROUP BY ps.seller_id
)
SELECT *
FROM tb_seller_rfv;